<a href="https://colab.research.google.com/github/SamDerden/Fraud_Detection_with-XAI_and_LLM/blob/main/%D0%B2%D0%B5%D0%B1%D0%B8%D0%BD%D0%B0%D1%80_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. Генерация синтетического датасета (с намеренными проблемами)
# ============================================================

def generate_dataset_with_issues(n_samples: int = 5000, random_state: int = 42) -> pd.DataFrame:
    """Создаём датасет, похожий на банковские транзакции, с несколькими проблемами."""
    rng = np.random.default_rng(random_state)

    user_ids = [f"usr_{i:05d}" for i in rng.integers(1, 800, n_samples)]
    transaction_ids = [f"txn_{i:06d}" for i in range(n_samples)]

    # Намеренно делаем несколько дубликатов transaction_id
    transaction_ids[10] = transaction_ids[5]
    transaction_ids[25] = transaction_ids[5]

    amounts = rng.lognormal(mean=8.0, sigma=1.2, size=n_samples).round(2)
    # Намеренно вставляем отрицательные и нулевые суммы
    amounts[50] = -1500.0
    amounts[51] = 0.0
    amounts[52] = -0.01

    # Timestamps
    base = datetime(2026, 1, 1)
    timestamps = [base + timedelta(minutes=int(x)) for x in rng.integers(0, 60*24*180, n_samples)]
    # Намеренно делаем будущую дату и пропуск
    timestamps[100] = datetime(2027, 6, 1)  # будущее
    timestamps[101] = None

    merchant_ids = [f"mrc_{i:04d}" for i in rng.integers(1, 300, n_samples)]
    # Пропуски в merchant_id
    for idx in [200, 201, 202]:
        merchant_ids[idx] = None

    device_ids = [f"dev_{i:04d}" for i in rng.integers(1, 500, n_samples)]
    countries = rng.choice(["RU", "KZ", "BY", "UZ", "TR", None], size=n_samples, p=[0.7, 0.1, 0.08, 0.05, 0.05, 0.02])

    # Метка фрода (дисбаланс)
    is_fraud = rng.choice([0, 1], size=n_samples, p=[0.92, 0.08])

    df = pd.DataFrame({
        "transaction_id": transaction_ids,
        "user_id": user_ids,
        "amount": amounts,
        "currency": "RUB",
        "timestamp": timestamps,
        "merchant_id": merchant_ids,
        "device_id": device_ids,
        "country": countries,
        "is_fraud": is_fraud,
    })

    # Ещё одна проблема: смешанный тип в amount (строка вместо числа)
    df["amount"] = df["amount"].astype(object)
    df.loc[300, "amount"] = "неизвестно"

    return df


# ============================================================
# 2. Набор проверок (минимум 5)
# ============================================================

class ValidationReport:
    def __init__(self):
        self.checks = []
        self.problems = []

    def add(self, name: str, status: str, details: str, severity: str = "warning"):
        self.checks.append({"name": name, "status": status, "details": details, "severity": severity})
        if status != "OK":
            self.problems.append({"name": name, "details": details, "severity": severity})

    def summary(self) -> str:
        lines = []
        lines.append("=" * 70)
        lines.append("ОТЧЁТ О ВАЛИДАЦИИ ДАТАСЕТА")
        lines.append("Кейс: Антифрод — детекция подозрительных транзакций")
        lines.append(f"Дата проверки: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        lines.append("=" * 70)
        lines.append("")

        lines.append("РЕЗУЛЬТАТЫ ПРОВЕРОК:")
        lines.append("-" * 70)
        for i, c in enumerate(self.checks, 1):
            icon = "✅" if c["status"] == "OK" else ("❌" if c["severity"] == "critical" else "⚠️")
            lines.append(f"{i}. {icon} [{c['status']}] {c['name']}")
            lines.append(f"   {c['details']}")
            lines.append("")

        lines.append("=" * 70)
        lines.append("СВОДКА ПРОБЛЕМ:")
        lines.append("-" * 70)

        if not self.problems:
            lines.append("Критических и предупреждающих проблем не обнаружено.")
        else:
            critical = [p for p in self.problems if p["severity"] == "critical"]
            warnings_ = [p for p in self.problems if p["severity"] == "warning"]

            if critical:
                lines.append(f"\n🔴 КРИТИЧЕСКИЕ ({len(critical)}):")
                for p in critical:
                    lines.append(f"  • {p['name']}: {p['details']}")

            if warnings_:
                lines.append(f"\n🟡 ПРЕДУПРЕЖДЕНИЯ ({len(warnings_)}):")
                for p in warnings_:
                    lines.append(f"  • {p['name']}: {p['details']}")

        lines.append("")
        lines.append("=" * 70)
        lines.append("РЕКОМЕНДАЦИИ:")
        lines.append("-" * 70)
        lines.append("1. Удалить или исправить дубликаты transaction_id.")
        lines.append("2. Отфильтровать/исправить отрицательные и нулевые суммы.")
        lines.append("3. Привести столбец amount к числовому типу, обработать некорректные значения.")
        lines.append("4. Заполнить или удалить пропуски в timestamp и merchant_id.")
        lines.append("5. Проверить записи с датами в будущем — возможные ошибки логирования.")
        lines.append("6. После очистки повторно запустить валидацию перед обучением модели.")
        lines.append("=" * 70)

        return "\n".join(lines)


def run_validation(df: pd.DataFrame) -> ValidationReport:
    report = ValidationReport()

    # ----- Проверка 1: Дубликаты transaction_id -----
    n_dupes = df["transaction_id"].duplicated().sum()
    if n_dupes > 0:
        report.add(
            name="Уникальность transaction_id",
            status="FAIL",
            details=f"Найдено {n_dupes} дубликатов transaction_id. Это критично для идентификации операций.",
            severity="critical"
        )
    else:
        report.add(
            name="Уникальность transaction_id",
            status="OK",
            details="Все transaction_id уникальны.",
            severity="info"
        )

    # ----- Проверка 2: Пропуски в ключевых полях -----
    key_cols = ["transaction_id", "user_id", "amount", "timestamp", "is_fraud"]
    missing = df[key_cols].isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        details = "; ".join([f"{col}: {cnt}" for col, cnt in missing.items()])
        report.add(
            name="Пропуски в ключевых полях",
            status="FAIL",
            details=f"Обнаружены пропуски → {details}",
            severity="critical"
        )
    else:
        report.add(
            name="Пропуски в ключевых полях",
            status="OK",
            details="В ключевых полях пропусков нет.",
            severity="info"
        )

    # ----- Проверка 3: Корректность сумм (amount) -----
    # Сначала пытаемся привести к числу
    amount_numeric = pd.to_numeric(df["amount"], errors="coerce")
    n_non_numeric = amount_numeric.isnull().sum() - df["amount"].isnull().sum()
    n_negative = (amount_numeric < 0).sum()
    n_zero = (amount_numeric == 0).sum()

    problems_amount = []
    if n_non_numeric > 0:
        problems_amount.append(f"{n_non_numeric} нечисловых значений")
    if n_negative > 0:
        problems_amount.append(f"{n_negative} отрицательных сумм")
    if n_zero > 0:
        problems_amount.append(f"{n_zero} нулевых сумм")

    if problems_amount:
        report.add(
            name="Корректность поля amount",
            status="FAIL",
            details="Проблемы: " + ", ".join(problems_amount),
            severity="critical"
        )
    else:
        report.add(
            name="Корректность поля amount",
            status="OK",
            details="Все суммы корректны (положительные числа).",
            severity="info"
        )

    # ----- Проверка 4: Валидность timestamp -----
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    n_bad_ts = ts.isnull().sum()
    now = pd.Timestamp.now()
    n_future = (ts > now).sum()

    if n_bad_ts > 0 or n_future > 0:
        details = []
        if n_bad_ts > 0:
            details.append(f"{n_bad_ts} некорректных/пустых дат")
        if n_future > 0:
            details.append(f"{n_future} дат в будущем")
        report.add(
            name="Валидность timestamp",
            status="FAIL",
            details="; ".join(details),
            severity="warning"
        )
    else:
        report.add(
            name="Валидность timestamp",
            status="OK",
            details="Все timestamps корректны и не находятся в будущем.",
            severity="info"
        )

    # ----- Проверка 5: Пропуски в важных, но не ключевых полях -----
    important_cols = ["merchant_id", "device_id", "country"]
    missing_imp = df[important_cols].isnull().sum()
    missing_imp = missing_imp[missing_imp > 0]
    if len(missing_imp) > 0:
        details = "; ".join([f"{col}: {cnt} ({cnt/len(df)*100:.1f}%)" for col, cnt in missing_imp.items()])
        report.add(
            name="Пропуски в дополнительных полях",
            status="WARN",
            details=details,
            severity="warning"
        )
    else:
        report.add(
            name="Пропуски в дополнительных полях",
            status="OK",
            details="Пропусков в merchant_id / device_id / country нет.",
            severity="info"
        )

    # ----- Проверка 6 (бонус): Дисбаланс классов -----
    fraud_rate = df["is_fraud"].mean()
    if fraud_rate < 0.01 or fraud_rate > 0.3:
        report.add(
            name="Дисбаланс классов (is_fraud)",
            status="WARN",
            details=f"Доля фрода = {fraud_rate:.1%}. Слишком экстремальный дисбаланс может усложнить обучение.",
            severity="warning"
        )
    else:
        report.add(
            name="Дисбаланс классов (is_fraud)",
            status="OK",
            details=f"Доля фрода = {fraud_rate:.1%} — приемлемый уровень для антифрод-задачи.",
            severity="info"
        )

    # ----- Проверка 7 (бонус): Базовая схема / типы -----
    expected_types = {
        "transaction_id": "object",
        "user_id": "object",
        "is_fraud": "int",
    }
    type_issues = []
    for col, expected in expected_types.items():
        actual = str(df[col].dtype)
        if expected == "int" and not actual.startswith("int"):
            type_issues.append(f"{col}: ожидался int, получен {actual}")
        elif expected == "object" and actual != "object":
            type_issues.append(f"{col}: ожидался object, получен {actual}")

    if type_issues:
        report.add(
            name="Соответствие типов данных",
            status="WARN",
            details="; ".join(type_issues),
            severity="warning"
        )
    else:
        report.add(
            name="Соответствие типов данных",
            status="OK",
            details="Основные типы данных соответствуют ожиданиям.",
            severity="info"
        )

    return report


# ============================================================
# 3. Запуск
# ============================================================

if __name__ == "__main__":
    print("Генерируем датасет с намеренными проблемами...")
    df = generate_dataset_with_issues(n_samples=5000)

    print(f"Размер датасета: {df.shape}")
    print(f"Колонки: {list(df.columns)}\n")

    print("Запускаем валидацию...\n")
    report = run_validation(df)

    # Печатаем отчёт
    text_report = report.summary()
    print(text_report)

    # Сохраняем отчёт в файл
    report_path = "validation_report_antifraud.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(text_report)

    print(f"\n📄 Отчёт сохранён в файл: {report_path}")

Генерируем датасет с намеренными проблемами...
Размер датасета: (5000, 9)
Колонки: ['transaction_id', 'user_id', 'amount', 'currency', 'timestamp', 'merchant_id', 'device_id', 'country', 'is_fraud']

Запускаем валидацию...

ОТЧЁТ О ВАЛИДАЦИИ ДАТАСЕТА
Кейс: Антифрод — детекция подозрительных транзакций
Дата проверки: 2026-10-24 10:19

РЕЗУЛЬТАТЫ ПРОВЕРОК:
----------------------------------------------------------------------
1. ❌ [FAIL] Уникальность transaction_id
   Найдено 2 дубликатов transaction_id. Это критично для идентификации операций.

2. ❌ [FAIL] Пропуски в ключевых полях
   Обнаружены пропуски → timestamp: 1

3. ❌ [FAIL] Корректность поля amount
   Проблемы: 1 нечисловых значений, 2 отрицательных сумм, 1 нулевых сумм

4. ⚠️ [FAIL] Валидность timestamp
   1 некорректных/пустых дат; 1 дат в будущем

5. ⚠️ [WARN] Пропуски в дополнительных полях
   merchant_id: 3 (0.1%); country: 109 (2.2%)

6. ✅ [OK] Дисбаланс классов (is_fraud)
   Доля фрода = 7.8% — приемлемый уровень для анти